# Make Bar Plot CSVs

In [ ]:
import os
import json
import csv
import numpy as np
from collections import defaultdict
from natsort import natsorted
from concurrent.futures import ProcessPoolExecutor, as_completed
from functools import partial


def _split_layers_into_groups(layer_folders):
    """Splits a sorted list of layer folders into early, mid, and late thirds."""
    n = len(layer_folders)
    if n == 0:
        return {}
    third = n // 3
    return {
        "early": layer_folders[:third],
        "mid":   layer_folders[third : 2 * third],
        "late":  layer_folders[2 * third :],
    }


def _process_sample_padded(sample_basename, layer_groups, npz_root, json_root, chunks_to_process):
    """
    Returns:
        dict[chunk][group_name][tag] -> float (per-sample mean attention value)
    """
    out = {}

    json_path = os.path.join(json_root, f"{sample_basename}.json")
    if not os.path.exists(json_path):
        return {}
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            tags_json = json.load(f)
    except Exception:
        return {}
    if not isinstance(tags_json, list) or len(tags_json) == 0:
        return {}
    token_tags = [tok.get('tag', 'OTHER') for tok in tags_json]

    for chunk in chunks_to_process:
        chunk_data = {}

        for group_name, layer_folders in layer_groups.items():
            layer_arrays = []
            for layer in layer_folders:
                npz_path = os.path.join(npz_root, layer, "attention_progression", f"{sample_basename}.npz")
                if not os.path.exists(npz_path):
                    continue
                try:
                    data = np.load(npz_path, allow_pickle=True)
                except Exception:
                    continue
                if chunk not in data:
                    continue
                arr = np.array(data[chunk], dtype=object)
                arr = np.where(arr == None, np.nan, arr).astype(float)
                layer_arrays.append(arr)

            if not layer_arrays:
                continue

            max_len = max(a.shape[0] for a in layer_arrays)
            if max_len == 0:
                continue

            padded = np.full((len(layer_arrays), max_len), np.nan, dtype=float)
            for i, a in enumerate(layer_arrays):
                padded[i, : a.shape[0]] = a

            with np.errstate(all='ignore'):
                per_token = np.nanmean(padded, axis=0)  # shape: (max_len,)

            eff_seq_len = min(len(token_tags), per_token.shape[0])
            if eff_seq_len == 0:
                continue

            tags_slice = token_tags[:eff_seq_len]
            tag_to_inds = {}
            for idx, t in enumerate(tags_slice):
                tag_to_inds.setdefault(t, []).append(idx)

            group_tag_means = {}
            for tag, inds in tag_to_inds.items():
                vals = per_token[inds]
                if np.all(np.isnan(vals)):
                    continue
                group_tag_means[tag] = float(np.nanmean(vals))

            if group_tag_means:
                chunk_data[group_name] = group_tag_means

        if chunk_data:
            out[chunk] = chunk_data

    return out


def extract_pos_tag_stats_to_csv(npz_root,
                                 json_root,
                                 chunks_to_process,
                                 output_csv_path,
                                 tags_to_include=None,
                                 layer_folders=None,
                                 sample_list=None,
                                 max_samples=None,
                                 max_workers=None,
                                 baseline_json_path=None,
                                 subtract_baseline=False,
                                 verbose=False):
    """
    Computes POS-tag averaged attention stats broken down by layer group
    (early / mid / late thirds) and saves them to a CSV file.

    Columns in output CSV:
        chunk, layer_group, tag, mean, std, median, count
        [, mean_baseline_subtracted]  <- only when subtract_baseline=True
    """

    # 1. Load baseline (if any)
    baseline = {}
    if baseline_json_path is not None:
        if not os.path.exists(baseline_json_path):
            raise FileNotFoundError(f"Baseline JSON not found: {baseline_json_path}")
        with open(baseline_json_path, 'r') as f:
            baseline = json.load(f)
        baseline = {k: float(v) for k, v in baseline.items()}

    # 2. Discover layers and split into groups
    if layer_folders is None:
        cand = [d for d in os.listdir(npz_root) if os.path.isdir(os.path.join(npz_root, d))]
        layer_folders = natsorted(cand)
    else:
        layer_folders = list(layer_folders)
    if not layer_folders:
        raise ValueError("No layer folders found in npz_root.")

    layer_groups = _split_layers_into_groups(layer_folders)

    if verbose:
        for g, layers in layer_groups.items():
            print(f"  [{g}] {len(layers)} layers: {layers[0]} … {layers[-1]}")

    # 3. Discover samples
    if sample_list is None:
        sample_list = []
        for layer in layer_folders:
            att_dir = os.path.join(npz_root, layer, "attention_progression")
            if os.path.isdir(att_dir):
                files = [f for f in os.listdir(att_dir) if f.endswith('.npz')]
                sample_list = natsorted([os.path.splitext(f)[0] for f in files])
                break
    if max_samples is not None:
        sample_list = sample_list[:max_samples]
    if not sample_list:
        raise ValueError("No samples found.")

    # 4. Configure workers
    if max_workers is None:
        try:
            import multiprocessing
            max_workers = min(8, multiprocessing.cpu_count() or 4)
        except Exception:
            max_workers = 4

    worker = partial(_process_sample_padded,
                     layer_groups=layer_groups,
                     npz_root=npz_root,
                     json_root=json_root,
                     chunks_to_process=chunks_to_process)

    # collector[chunk][group_name][tag] -> list of per-sample means
    GROUP_NAMES = ["early", "mid", "late"]
    collector = {
        chunk: {g: defaultdict(list) for g in GROUP_NAMES}
        for chunk in chunks_to_process
    }

    if verbose:
        print(f"Processing {len(sample_list)} samples with {max_workers} workers...")
        print(f"Target file: {output_csv_path}")

    # 5. Execute parallel processing
    with ProcessPoolExecutor(max_workers=max_workers) as ex:
        futures = {ex.submit(worker, sb): sb for sb in sample_list}
        for fut in as_completed(futures):
            sb = futures[fut]
            try:
                res = fut.result()
            except Exception as e:
                if verbose:
                    print(f"[error] sample {sb} -> {e}")
                continue
            if not res:
                continue
            for chunk, group_data in res.items():
                for group_name, tagmap in group_data.items():
                    for tag, val in tagmap.items():
                        if np.isnan(val):
                            continue
                        collector[chunk][group_name][tag].append(float(val))

    # 6. Aggregate stats and write CSV
    output_dir = os.path.dirname(output_csv_path)
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)

    available_tags = [
        "SOG", "FRUIT_INTRO", "FRUIT_CONCEPT",
        "FIRST_HANDOFF", "OTHER_HANDOFF",
        "MATH_INTRO", "MATH_ANSWER", "POSTAMBLE",
    ]

    rows_written = 0
    with open(output_csv_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)

        header = ['chunk', 'layer_group', 'tag', 'mean', 'std', 'median', 'count']
        if subtract_baseline:
            header.append('mean_baseline_subtracted')
        writer.writerow(header)

        for chunk in chunks_to_process:
            for group_name in GROUP_NAMES:
                for tag in available_tags:
                    if tags_to_include is not None and tag not in tags_to_include:
                        continue

                    vals = collector[chunk][group_name][tag]
                    arr = np.array(vals, dtype=float)
                    if arr.size == 0:
                        continue

                    mean_val   = float(np.nanmean(arr))
                    std_val    = float(np.nanstd(arr)) if arr.size > 1 else 0.0
                    median_val = float(np.nanmedian(arr))
                    count_val  = int(np.count_nonzero(~np.isnan(arr)))

                    row = [chunk, group_name, tag, mean_val, std_val, median_val, count_val]

                    if subtract_baseline:
                        adjusted = mean_val - baseline.get(tag, 0.0)
                        row.append(adjusted)

                    writer.writerow(row)
                    rows_written += 1

    if verbose:
        print(f"Done. Wrote {rows_written} rows to {output_csv_path}")

    return collector

In [ ]:
npz_root = '<RUN_ROOT>/otat/FrMaSc/lov_7b'
json_root = '<REPO_ROOT>/data/other_gemini_scaling_stuff/batch_tagging/FrMaSc_pos_tags/lov_7b/fruit_math'


chunks_to_process=[
        "currently_generating_token__attends_to__image",
        "currently_generating_token__attends_to__text",
        "currently_generating_token__attends_to__instruction",
        "currently_generating_token__attends_to__previously_generating_tokens",

    ]

# tags_to_include=["SOG", "SPORT_INTRO", "SPORT_CONCEPT", "FIRST_HANDOFF", "OTHER_HANDOFF", "MATH_INTRO", "MATH_ANSWER", "POSTAMBLE"]
output_csv_path = '<REPO_ROOT>/data/bar_plot_csv/bar_plot_csv_data_layerGrouped/fruit_math/lov7B.csv'



extract_pos_tag_stats_to_csv(npz_root,
                                 json_root,
                                 chunks_to_process,
                                 output_csv_path,
                                 tags_to_include=None,
                                 layer_folders=None,
                                 sample_list=None,
                                 max_samples=None,
                                 max_workers=None,
                                 baseline_json_path=None,
                                 subtract_baseline=False,
                                 verbose=True)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def plot_layer_group_attention(
    per_tag_csv,
    tag_order=None,
    tag_rename=None,
    colors=None,
    chunk_to_stack=None,
    layer_groups=("early", "mid", "late"),
    group_styles=None,
    normalize=True,           # subtract per-chunk-group global mean from the CSV itself
    xlabel="",
    ylabel="Normalized Attention",
    title="",
    figsize=(8, 3.5),
    bar_width=0.08,
    group_gap=0.04,
    linewidth=0.6,
    tick_fontsize=13,
    label_fontsize=12,
    title_fontsize=10,
    plot_std=False,
    show_spines=False,
    legend_loc="upper right",
    legend_fontsize=10,
    legend_ncol=2,
    legend_frame=False,
    merge_tags=None,
    show_table=False
):
    if chunk_to_stack is None:
        chunk_to_stack = {
            "currently_generating_token__attends_to__image":                        "Image",
            "currently_generating_token__attends_to__text":                         "Text",
            "currently_generating_token__attends_to__instruction":                  "Instruction",
            "currently_generating_token__attends_to__previously_generating_tokens": "Previous",
        }

    STACKS = list(dict.fromkeys(chunk_to_stack.values()))
    GROUPS = list(layer_groups)

    if colors is None:
        raise ValueError("Provide `colors` dict keyed by stack name.")
    missing = set(STACKS) - set(colors)
    if missing:
        raise ValueError(f"Missing colors for stacks: {missing}")

    if group_styles is None:
        group_styles = {
            "early": dict(hatch="",    alpha=1.00),
            "mid":   dict(hatch="//",  alpha=0.85),
            "late":  dict(hatch="xx",  alpha=0.70),
        }

    # ── load & filter ─────────────────────────────────────────────────────────
    df = pd.read_csv(per_tag_csv)
    df = df[df["chunk"].isin(chunk_to_stack) & df["layer_group"].isin(GROUPS)].copy()
    df["stack"] = df["chunk"].map(chunk_to_stack)

    # ── normalize: subtract mean-of-means per (chunk, layer_group) ────────────
    if normalize:
        global_means = (
            df.groupby(["chunk", "layer_group"])["mean"]
            .mean()
            .rename("global_mean")
            .reset_index()
        )
        df = df.merge(global_means, on=["chunk", "layer_group"])
        df["mean"] = df["mean"] - df["global_mean"]

    # ── pivot ─────────────────────────────────────────────────────────────────
    mean_pivot = df.pivot_table(
        index="tag", columns=["stack", "layer_group"], values="mean", aggfunc="mean"
    )
    std_pivot = df.pivot_table(
        index="tag", columns=["stack", "layer_group"], values="std", aggfunc="mean"
    )

    mean_pivot = mean_pivot.reset_index()
    std_pivot  = std_pivot.reset_index()

    # ── fix MultiIndex columns after reset_index ───────────────────────────────
    mean_pivot.columns = [
        c[0] if c[1] == "" else c          # ("tag", "") → "tag"; ("Image","early") stays a tuple
        for c in mean_pivot.columns
    ]
    std_pivot.columns = [
        c[0] if c[1] == "" else c
        for c in std_pivot.columns
    ]

    # ── merge tags ────────────────────────────────────────────────────────────
    if merge_tags:
        cols = [(s, g) for s in STACKS for g in GROUPS]
        # keep only cols that exist
        cols = [c for c in cols if c in mean_pivot.columns]
        for keep_tag, remove_tag in merge_tags:
            if keep_tag in mean_pivot["tag"].values and remove_tag in mean_pivot["tag"].values:
                k_m = mean_pivot.loc[mean_pivot["tag"] == keep_tag,  cols].values
                r_m = mean_pivot.loc[mean_pivot["tag"] == remove_tag, cols].values
                mean_pivot.loc[mean_pivot["tag"] == keep_tag, cols] = (k_m + r_m) / 2.0

                k_s = std_pivot.loc[std_pivot["tag"] == keep_tag,  cols].values
                r_s = std_pivot.loc[std_pivot["tag"] == remove_tag, cols].values
                std_pivot.loc[std_pivot["tag"] == keep_tag, cols] = (k_s + r_s) / 2.0

                mean_pivot = mean_pivot[mean_pivot["tag"] != remove_tag]
                std_pivot  = std_pivot[std_pivot["tag"]  != remove_tag]
            else:
                print(f"Warning: could not merge '{remove_tag}' into '{keep_tag}'.")

    # ── ordering ──────────────────────────────────────────────────────────────
    if tag_order is not None:
        available = set(mean_pivot["tag"].unique())
        filtered  = [t for t in tag_order if t in available]
        mean_pivot["tag"] = pd.Categorical(mean_pivot["tag"], filtered, ordered=True)
        std_pivot["tag"]  = pd.Categorical(std_pivot["tag"],  filtered, ordered=True)
        mean_pivot = mean_pivot.sort_values("tag").dropna(subset=["tag"])
        std_pivot  = std_pivot.sort_values("tag").dropna(subset=["tag"])

    n_tags   = len(mean_pivot)
    n_groups = len(GROUPS)
    cluster_width  = n_groups * bar_width
    total_per_tag  = len(STACKS) * cluster_width + (len(STACKS) - 1) * group_gap
    x = np.arange(n_tags) * (total_per_tag + 0.25)

    # ── plot ──────────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=figsize)
    legend_handles = {}

    for si, stack in enumerate(STACKS):
        stack_offset = si * (cluster_width + group_gap)
        for gi, group in enumerate(GROUPS):
            col = (stack, group)
            if col not in mean_pivot.columns:
                continue

            y    = mean_pivot[col].values.astype(float)
            yerr = std_pivot[col].values.astype(float) if plot_std else None
            style = group_styles.get(group, {})

            ax.bar(
                x + stack_offset + gi * bar_width,
                y,
                bar_width,
                color=colors[stack],
                linewidth=linewidth,
                yerr=yerr,
                error_kw=dict(elinewidth=0.6, capsize=1.5, capthick=0.6) if plot_std else None,
                **style,
            )

            if stack not in legend_handles:
                legend_handles[stack] = plt.Rectangle(
                    (0, 0), 1, 1, fc=colors[stack], ec="none", label=stack,
                )
            if group not in legend_handles:
                legend_handles[group] = plt.Rectangle(
                    (0, 0), 1, 1,
                    fc="grey",
                    hatch=style.get("hatch", ""),
                    alpha=style.get("alpha", 1.0),
                    ec="white",
                    label=group.capitalize(),
                )

    # ── axes ──────────────────────────────────────────────────────────────────
    ax.axhline(0, linewidth=0.6, color="black")

    xtick_pos   = x + total_per_tag / 2 - bar_width / 2
    raw_labels  = mean_pivot["tag"].astype(str).tolist()
    final_labels = [
        (tag_rename.get(t, t) if tag_rename else t).replace(" ", "\n")
        for t in raw_labels
    ]
    ax.set_xticks(xtick_pos)
    ax.set_xticklabels(final_labels, rotation=0, ha="center", fontsize=tick_fontsize)
    ax.tick_params(axis="y", labelsize=tick_fontsize)
    ax.set_xlabel(xlabel, fontsize=label_fontsize)
    ax.set_ylabel(ylabel, fontsize=label_fontsize)
    ax.set_title(title, fontsize=title_fontsize)

    ordered_handles = (
        [legend_handles[s] for s in STACKS if s in legend_handles] +
        [legend_handles[g] for g in GROUPS  if g in legend_handles]
    )
    ax.legend(
        handles=ordered_handles,
        loc=legend_loc,
        fontsize=legend_fontsize,
        ncol=legend_ncol,
        frameon=legend_frame,
        handlelength=1.2,
        handletextpad=0.4,
        columnspacing=0.8,
    )

    if not show_spines:
        for spine in ["top", "right", "left", "bottom"]:
            ax.spines[spine].set_visible(False)
        ax.tick_params(axis="both", length=0)

    plt.tight_layout(pad=0.4)
    plt.show()

    if show_table:
        from tabulate import tabulate

# inside the function, add show_table=False to the signature, then at the end:

    if show_table:
        # Build a readable table: rows=tags, cols=(stack, group)
        rows = []
        for _, row in mean_pivot.iterrows():
            tag = str(row["tag"])
            entry = [tag]
            for stack in STACKS:
                for group in GROUPS:
                    col = (stack, group)
                    val = row[col] if col in mean_pivot.columns else float("nan")
                    entry.append(f"{val:+.4f}" if not np.isnan(val) else "—")
            rows.append(entry)

        headers = ["Tag"] + [f"{s}\n{g}" for s in STACKS for g in GROUPS]

        print(tabulate(
            rows,
            headers=headers,
            # tablefmt="rounded_outline",
            # colalign=["left"] + ["center"] * (len(STACKS) * len(GROUPS)),
        ))

    return fig

In [ ]:
fig = plot_layer_group_attention(
    per_tag_csv="<REPO_ROOT>/data/bar_plot_csv/bar_plot_csv_data_layerGrouped/fruit_math/q25vl7B.csv",  # new layer-group CSV
    tag_rename={
        "OTHER_HANDOFF": "HANDOFF",
    },
    chunk_to_stack={
        "currently_generating_token__attends_to__image":                   "Image",
        "currently_generating_token__attends_to__text":                    "Text",
        "currently_generating_token__attends_to__instruction":                  "Instruction",
        "currently_generating_token__attends_to__previously_generating_tokens": "Previous",
    },

    tag_order = [
        "SOG", "FRUIT_INTRO", "FRUIT_CONCEPT",
        "FIRST_HANDOFF", "OTHER_HANDOFF",
        "MATH_INTRO", "MATH_ANSWER", "POSTAMBLE",
    ],
    colors = {
    "Image": "#FFA500",
    "Text": "#267E59",
    "Instruction": "#D25B5B",
    "Previous": "#C01BA7",
    },

    # Optional: override default hatch/alpha per layer group
    group_styles={
        "early": dict(hatch="",   alpha=1.00),
        "mid":   dict(hatch="//", alpha=0.85),
        "late":  dict(hatch="xx", alpha=0.70),
    },
    figsize=(18, 4),
    bar_width=0.07,
    group_gap=0.05,
    ylabel="Mean attention − global mean",
    merge_tags=[("FRUIT_CONCEPT", "FIRST_HANDOFF")],
    show_table=True,
);

#save as pdf
fig.savefig("layer_group_attention_qvl7b.pdf", bbox_inches="tight")

In [ ]:
fig = plot_layer_group_attention(
    per_tag_csv="<REPO_ROOT>/data/bar_plot_csv/bar_plot_csv_data_layerGrouped/fruit_math/lov7B.csv",  # new layer-group CSV
    tag_rename={
        "OTHER_HANDOFF": "HANDOFF",
    },
    chunk_to_stack={
        "currently_generating_token__attends_to__image":                   "Image",
        "currently_generating_token__attends_to__text":                    "Text",
        "currently_generating_token__attends_to__instruction":                  "Instruction",
        "currently_generating_token__attends_to__previously_generating_tokens": "Previous",
    },

    tag_order = [
        "SOG", "FRUIT_INTRO", "FRUIT_CONCEPT",
        "FIRST_HANDOFF", "OTHER_HANDOFF",
        "MATH_INTRO", "MATH_ANSWER", "POSTAMBLE",
    ],
    colors = {
    "Image": "#FFA500",
    "Text": "#267E59",
    "Instruction": "#D25B5B",
    "Previous": "#C01BA7",
    },

    # Optional: override default hatch/alpha per layer group
    group_styles={
        "early": dict(hatch="",   alpha=1.00),
        "mid":   dict(hatch="//", alpha=0.85),
        "late":  dict(hatch="xx", alpha=0.70),
    },
    figsize=(18, 4),
    bar_width=0.07,
    group_gap=0.05,
    ylabel="Mean attention − global mean",
    merge_tags=[("FRUIT_CONCEPT", "FIRST_HANDOFF")],
    show_table=True,
);

fig.savefig("layer_group_attention_lov7b.pdf", bbox_inches="tight")